# Распознавание объектов и проверка описаний

Модели извлекают области из JPEG. Одна учится различать фигуры по направлениям рёбер на кропах. Вторая подгоняет проекции кубов, сфер и цилиндров, чтобы разделить слившиеся области. Текст превращается в условия над таблицей объектов. Парные подписи оцениваются по общим гипотезам о сцене.

Через Add Input подключите данные соревнования и частный Dataset с архивом `visual-reasoning-silhouettes.zip`. Установите Internet Off. При наличии двух T4 каждая обучает свою модель.

In [ ]:
RUN_FINAL = True
BATCH_SIZE = 64  # сцен на одну модель
SEED = 42
SCHEDULE_EPOCHS = 20
FINAL_EPOCHS = {"crops": 7, "silhouettes": 19}
DATA_DIR = None
SOURCE = None

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
from zipfile import ZipFile

import catboost
import cv2
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import accuracy_score, roc_auc_score
import torch

kaggle = Path("/kaggle/input").is_dir()
inputs = Path("/kaggle/input") if kaggle else Path.cwd()
output = Path("/kaggle/working") if kaggle else Path.cwd()

def locate(explicit, candidates, setting):
    if explicit is not None:
        return Path(explicit).resolve(strict=True)
    found = sorted({p.resolve() for p in candidates})
    if len(found) != 1:
        raise ValueError(f"Укажите {setting} в первой ячейке. Найдено: {found}")
    return found[0]

data = locate(DATA_DIR, (
    p.parent for p in inputs.rglob("train.csv")
    if (p.parent / "train").is_dir() and (p.parent / "val.csv").is_file()
), "DATA_DIR")
archives = list(inputs.rglob("visual-reasoning-silhouettes.zip"))
expanded = [
    p.parent.parent for p in inputs.rglob("region_shape_classifier.py")
    if (p.parent.parent / "scripts/pixel_reasoning.py").is_file()
]
source = locate(SOURCE, archives or expanded, "SOURCE")
project = Path(tempfile.mkdtemp(prefix="visual_reasoning_silhouettes_")).resolve()
if source.is_file():
    with ZipFile(source) as archive:
        if any(not (project / name).resolve().is_relative_to(project) for name in archive.namelist()):
            raise ValueError("Недопустимый путь внутри архива")
        archive.extractall(project)
else:
    for folder in ("scripts", "src"):
        shutil.copytree(source / folder, project / folder, ignore=shutil.ignore_patterns("__pycache__"))

gpu_count = torch.cuda.device_count()
visible = os.environ.get("CUDA_VISIBLE_DEVICES", ",".join(map(str, range(gpu_count)))).split(",")
print(f"Данные: {data} | GPU: {gpu_count} | OpenCV: {cv2.__version__} | CatBoost: {catboost.__version__}")

## Обучение и предсказания

`RUN_FINAL=True` обучает модели с нуля на train и создаёт `final_submission.csv` в порядке sample_submission. Test открывается для предсказаний уже после завершения обучения каждой модели. `RUN_FINAL=False` использует val для выбора эпохи и выводит AUC и accuracy на val.

In [ ]:
start = time.monotonic()
reports = Path(tempfile.mkdtemp(prefix="silhouettes_run_", dir=output)).resolve()
variants = {"crops": "--crop-shapes", "silhouettes": "--primitives"}

def train_worker(item):
    index, (name, flag) = item
    receipt = reports / f"{name}.json"
    epochs = FINAL_EPOCHS[name] if RUN_FINAL else SCHEDULE_EPOCHS
    command = [
        sys.executable, "-u", "-m", "scripts.pixel_reasoning",
        "--data", str(data), "--epochs", str(epochs), "--schedule-epochs", str(SCHEDULE_EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--seed", str(SEED), "--report", str(receipt),
        "--anchors", "--posterior", "joint", flag,
    ]
    if RUN_FINAL:
        command += ["--final", "--output", str(reports / f"{name}.csv")]
    env = {**os.environ, "PYTHONHASHSEED": str(SEED), "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
           "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"}
    if gpu_count:
        env["CUDA_VISIBLE_DEVICES"] = visible[index % gpu_count]
    log_path = reports / f"{name}.log"
    print(f"{name}: эпохи {epochs}, журнал {log_path}", flush=True)
    with log_path.open("w", encoding="utf-8") as stream, subprocess.Popen(
        command, cwd=project, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    ) as process:
        for line in process.stdout:
            stream.write(line)
            stream.flush()
            if line.startswith(("Extracting", "Training CatBoost", "Features ready")):
                print(f"{name} | {line.strip()}", flush=True)
            elif line.startswith('{"epoch":'):
                row = json.loads(line)
                metric = f", AUC {row['auc']:.5f}" if "auc" in row else ""
                print(f"{name} | эпоха {row['epoch']}/{epochs}, loss {row['loss']:.4f}{metric}", flush=True)
        status = process.wait()
    if status:
        tail = "\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-24:])
        raise RuntimeError(f"{name} завершилась с кодом {status}\n{tail}")
    return name, json.loads(receipt.read_text(encoding="utf-8"))

jobs = list(enumerate(variants.items()))
with ThreadPoolExecutor(max_workers=2 if gpu_count >= 2 else 1) as pool:
    receipts = dict(pool.map(train_worker, jobs))
print(pd.DataFrame({name: {"эпохи": run["configuration"]["epochs"],
                          "устройство": run["device"], "минуты": run["minutes"]}
                    for name, run in receipts.items()}).T)

if RUN_FINAL:
    sample = pd.read_csv(data / "sample_submission.csv")
    logits = []
    for name in variants:
        predictions = pd.read_csv(reports / f"{name}.csv")
        if list(predictions.columns) != ["id", "label"] or not predictions.id.equals(sample.id):
            raise ValueError(f"{name}: неверный формат или порядок id")
        if not ((predictions.label > 0) & (predictions.label < 1)).all():
            raise ValueError(f"{name}: неверные вероятности")
        logits.append(logit(predictions.label.to_numpy()))
    sample["label"] = expit(np.mean(logits, axis=0))
    destination = output / "final_submission.csv"
    staged = reports / "final_submission.csv"
    sample.to_csv(staged, index=False)
    staged.replace(destination)
    print(f"{destination} | строк: {len(sample)}")
else:
    val = pd.read_csv(data / "val.csv")
    scores = {}
    for name in variants:
        with np.load(reports / f"{name}.npz") as report:
            if not np.array_equal(report["ids"], val.id.to_numpy()):
                raise ValueError(f"{name}: порядок id не совпадает с val.csv")
            scores[name] = report["joint"]
    scores["ensemble"] = np.mean(list(scores.values()), axis=0)
    metrics = {}
    for name, values in scores.items():
        row = {"AUC": roc_auc_score(val.label, values),
               "accuracy": accuracy_score(val.label, values >= 0)}
        for category, group in val.groupby("category"):
            row[category] = roc_auc_score(group.label, values[group.index])
        metrics[name] = row
    display(pd.DataFrame.from_dict(metrics, orient="index").round(6))
print(f"Время двух моделей: {(time.monotonic() - start) / 60:.2f} мин. Отчёты: {reports}")